# Dengue Breeding Site Scanner: train the first model (Kaggle version)

This is the same notebook as the Colab one, adapted for **Kaggle**, which also lends free GPUs (about 30 hours a week).

**Before you start** (see the Kaggle guide in `BUILD-WALKTHROUGH.md` for screenshots of each step)
1. Your Kaggle account is **phone-verified** (needed for GPU and internet).
2. `merged-v1.zip` is uploaded as a **Kaggle Dataset** and added to this notebook (right panel → **Input**).
3. In the right panel **Settings**: **Accelerator = GPU T4 x2** (or GPU P100) and **Internet = On**.

Then run the cells from top to bottom (**Shift + Enter**), reading the text cell above each code cell first.

**Two ways to run the long training step**
- **Interactive** (this page): keep the tab open for about an hour.
- **In the background:** click **Save Version → Save & Run All (Commit)**. Kaggle runs every cell on its own (up to 12 hours), even if you close the tab, and keeps all outputs.

| Step | What happens |
|---|---|
| 1 | Check the GPU |
| 2 | Find the dataset |
| 3 | Settings |
| 4 | Copy the dataset to a writable disk |
| 5 | Install YOLO |
| 6 | Look at the data |
| 7 | Train |
| 8 | Read the training charts |
| 9 | Test on the exam set |
| 10 | Find what it missed, save the model |

## Step 1: Check the GPU

Training means doing millions of small multiplications. A **GPU** does thousands of them at once, so training that would take days on a laptop CPU takes about an hour.

**First:** right panel → **Settings → Accelerator → GPU T4 x2** (or **GPU P100**). Kaggle restarts the session when you change it.

Run the cell below. It should print a table naming a **Tesla T4** (two of them with T4 x2) or a **Tesla P100**. If it says the command was not found, the GPU is not on.

In [ ]:
!nvidia-smi

## Step 2: Find the dataset

On Kaggle, datasets you add to a notebook appear as read-only folders under **`/kaggle/input/`**. Kaggle usually unzips an uploaded zip for you, so we search for our `data.yaml` instead of guessing the exact folder name.

If the cell finds nothing, the dataset is not added yet: right panel → **Input → Add Input** → search for your dataset → **Add**.

In [ ]:
import glob, os
found = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
print('data.yaml found at:', found)
print('zip files found at:', zips)
assert found or zips, 'No dataset found under /kaggle/input. Add it with Input -> Add Input.'

## Step 3: Settings

Everything you might want to change lives in this one cell.

| Setting | Meaning | Why this value |
|---|---|---|
| `MODEL` | Which YOLO to start from | `yolo11n.pt` is the smallest ("n" = nano): fast to train, fast on phones |
| `EPOCHS` | How many times the model studies every training photo | 50 is a solid first run |
| `IMGSZ` | Image size the model sees | Our photos are already 640×640 |
| `BATCH` | Photos studied at once | 16 fits in a T4's memory |
| `PATIENCE` | Stop early if the score hasn't improved for this many epochs | Saves time once learning stalls |
| `SEED` | Fixes the randomness | Same settings → same result |

`/kaggle/working/` is the only folder Kaggle keeps after the session, so results go there. If training crashes with **"CUDA out of memory"**, change `BATCH` to 8.

In [ ]:
RUN_NAME = 'baseline-v1'

MODEL    = 'yolo11n.pt'
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16
PATIENCE = 15
SEED     = 42

DATA_DIR = '/tmp/dataset'          # writable copy of the dataset (Kaggle's input folder is read-only)
RUNS_DIR = '/kaggle/working/runs'  # Kaggle keeps everything in /kaggle/working after the session
print('Settings ready. Results will be saved to:', RUNS_DIR)

## Step 4: Copy the dataset to a writable disk

`/kaggle/input/` is **read-only**, but YOLO likes to write small cache files next to the labels. So we copy the dataset to `/tmp`, a fast local disk we can write to. (We don't copy it into `/kaggle/working/`: that folder is saved after the session, and a 1 GB dataset copy would just waste space.)

You should see **4321 train, 423 valid, 235 test**, the same numbers as our merge report.

In [ ]:
import shutil, zipfile, pathlib

shutil.rmtree(DATA_DIR, ignore_errors=True)
if found:   # Kaggle already unzipped it: copy the folder that holds data.yaml
    shutil.copytree(os.path.dirname(found[0]), DATA_DIR)
else:       # still a zip: unzip it
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATA_DIR)

DATA_YAML = f'{DATA_DIR}/data.yaml'
for split in ['train', 'valid', 'test']:
    n = len(list(pathlib.Path(DATA_DIR, split, 'images').glob('*')))
    print(f'{split:5}: {n} images')
print()
print(open(DATA_YAML).read())

## Step 5: Install YOLO

**Ultralytics** is the library that contains YOLO. One install gives us training, testing and prediction.

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## Step 6: Look at the data before training

Always look before you train. This draws the **labels** (the boxes a person drew) on 6 random training photos.

Check: is each box around the whole object? Do the class names match what you see? If many look wrong, stop here and fix the data: a model can only be as good as its labels.

In [ ]:
import random, yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

names = yaml.safe_load(open(DATA_YAML))['names']
COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6']

def draw_labels(img_path):
    # Draw the boxes from the YOLO label file that belongs to this image.
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    d = ImageDraw.Draw(img)
    label_path = str(img_path).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(label_path):
        for line in open(label_path):
            c, x, y, w, h = line.split()
            c, x, y, w, h = int(c), float(x) * W, float(y) * H, float(w) * W, float(h) * H
            d.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline=COLORS[c], width=4)
            d.text((x - w / 2 + 4, y - h / 2 + 4), names[c], fill=COLORS[c])
    return img

random.seed(1)
samples = random.sample(list(pathlib.Path(DATA_DIR, 'train', 'images').glob('*')), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, p in zip(axes.flat, samples):
    ax.imshow(draw_labels(p)); ax.set_title(p.name[:40], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## Step 7: Train

### Transfer learning: we don't start from zero
`yolo11n.pt` has already been trained on a big public dataset (COCO: 80 everyday classes such as cars, people and bottles). It already knows edges, shapes and textures. We **fine-tune** it: it keeps that knowledge and learns our 7 classes on top. That is why a few thousand photos are enough.

### What one epoch is
In one **epoch** the model looks at every training photo once:
1. It guesses boxes for a batch of 16 photos.
2. It compares its guesses with our labels and computes the **loss** (how wrong it was).
3. It nudges its weights to be a little less wrong.

After each epoch it takes a quiz on the **valid** split and prints the scores.

### What to watch in the output
- `box_loss`, `cls_loss`: should **go down** over the epochs (it is getting less wrong).
- `mAP50`: should **go up** (0 = useless, 1 = perfect).

`device=0` uses one GPU. With "T4 x2" the second GPU sits idle; using both needs extra setup that often breaks inside notebooks, and one T4 is enough for this dataset.

If the session stops mid-training, rerun Steps 1 to 5, then the "resume" cell after this one. (A **Save & Run All** commit avoids this: it runs in the background.)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)   # downloads the pretrained starting point (needs Internet = On)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    device=0,
    project=RUNS_DIR,
    name=RUN_NAME,
    exist_ok=True,
)
SAVE_DIR = pathlib.Path(model.trainer.save_dir)
BEST = SAVE_DIR / 'weights' / 'best.pt'
print('Best model saved at:', BEST)

### Only if the session stopped during training
Set `RESUME = True` in the cell below and run it instead of the training cell above. Leave it `False` otherwise: then the cell does nothing, so **Run All** and **Save & Run All** pass straight through it. It continues from the last saved epoch, as long as `/kaggle/working/runs` still has it (in the same session, or a commit's output added as input).

In [ ]:
RESUME = False   # change to True ONLY if training was cut off and you want to continue it

if RESUME:
    SAVE_DIR = pathlib.Path(RUNS_DIR, RUN_NAME)
    model = YOLO(str(SAVE_DIR / 'weights' / 'last.pt'))
    results = model.train(resume=True)
    BEST = SAVE_DIR / 'weights' / 'best.pt'
    print('Best model saved at:', BEST)
else:
    print('Skipped: training finished normally, nothing to resume.')

## Step 8: Read the training charts

`results.png` shows how training went, epoch by epoch:
- **Top row and loss charts**: `train/box_loss`, `train/cls_loss` should slope **down**.
- **Validation losses** (`val/...`) should also go down. If they go **up** while the training losses keep falling, the model is **overfitting**: memorising the training photos instead of learning.
- **metrics/mAP50(B)** should climb and flatten out.

The **confusion matrix** shows which classes get mixed up: read a row as "the real class was X", and the columns as "the model said Y". Big numbers off the diagonal = classes it confuses. The **background** column = objects it missed completely.

In [ ]:
from IPython.display import Image as Show, display
display(Show(filename=str(SAVE_DIR / 'results.png'), width=1000))
display(Show(filename=str(SAVE_DIR / 'confusion_matrix_normalized.png'), width=700))

## Step 9: Test on the exam set

Now the real exam: the **test** split, which the model never saw during training.

| Score | Question it answers |
|---|---|
| **Precision** | Of the boxes the model drew, how many were right? Low = false alarms |
| **Recall** | Of the real objects, how many did it find? Low = missed breeding sites |
| **mAP50** | One overall score for finding objects with the right class and a box that overlaps the real one by at least 50% |

For a dengue scanner, **recall matters most**: a missed breeding site is worse than a false alarm. Look at the per-class table: expect Cup and Plastic-Container (only about 110 boxes each) to score lowest.

**Copy the summary line at the end into `docs/results.md` in the project.**

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=DATA_YAML, split='test', imgsz=IMGSZ, batch=BATCH, plots=True,
                         project=RUNS_DIR, name=f'{RUN_NAME}-test', exist_ok=True)

box = metrics.box
print(f"\n{'Class':<20}{'Precision':>10}{'Recall':>10}{'mAP50':>10}")
for i, c in enumerate(box.ap_class_index):
    print(f"{metrics.names[int(c)]:<20}{box.p[i]:>10.3f}{box.r[i]:>10.3f}{box.ap50[i]:>10.3f}")
print(f"{'ALL':<20}{box.mp:>10.3f}{box.mr:>10.3f}{box.map50:>10.3f}")

print('\nRow for docs/results.md:')
print(f"| (date) | (you) | merged v1 | {MODEL.replace('.pt', '')} | {EPOCHS} | {box.mp:.3f} | {box.mr:.3f} | {box.map50:.3f} | first baseline |")

## Step 10: Find what it missed, then save the model

Scores tell us **how much** it misses; pictures tell us **what** it misses. This cell runs the model on every test photo and lists the photos where it found **fewer objects than the labels say**. Then it shows the 6 worst: the labels a person drew on the left, the model's guesses on the right.

These misses are the most useful thing this notebook produces. They tell the team which photos to take next (Step 4 of our plan: find misses, add photos like them, retrain).

In [ ]:
test_images = sorted(pathlib.Path(DATA_DIR, 'test', 'images').glob('*'))
misses = []
for p in test_images:
    label_path = str(p).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    truth = sum(1 for _ in open(label_path)) if os.path.exists(label_path) else 0
    found = len(best_model.predict(str(p), conf=0.25, verbose=False)[0].boxes)
    if found < truth:
        misses.append((truth - found, truth, found, p))
misses.sort(reverse=True)
print(f'{len(misses)} of {len(test_images)} test photos have objects the model missed.')

for missed, truth, found, p in misses[:6]:
    pred = best_model.predict(str(p), conf=0.25, verbose=False)[0].plot()[:, :, ::-1]   # BGR -> RGB
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 6))
    a.imshow(draw_labels(p)); a.set_title(f'Labels: {truth} objects'); a.axis('off')
    b.imshow(pred); b.set_title(f'Model found {found}'); b.axis('off')
    plt.suptitle(p.name[:60], fontsize=9); plt.tight_layout(); plt.show()

### Save the model

`best.pt` is the trained model: the epoch with the best validation score. It is in `/kaggle/working/runs/baseline-v1/weights/`.

**To download it:**
- Interactive session: right panel → **Output** → open `runs/baseline-v1/weights/` → click the **⋮** next to `best.pt` → **Download**.
- After a **Save & Run All** commit: open the notebook's version page → **Output** tab → download `best.pt`.

Put it in the project's **`models/`** folder on your computer; the web demo will load it from there.

Finally, write down in the team log:
1. The results row from Step 9 (into `docs/results.md`).
2. Which kinds of objects it missed in Step 10 (for example "small bottles in shade", "cups far away").

In [ ]:
print('Your trained model:')
print(BEST)
print('\nAll charts and results for this run (download them from the Output panel):')
print(SAVE_DIR)